# Varredura de perdas e pesos — validação cruzada, sem teste

Compara várias combinações de perdas e pesos com **o mesmo protocolo e os mesmos *folds*** dos experimentos: 5 *folds* nas 37
imagens de treino, até 200 épocas, parada após 20 validações sem melhora, ScribblePrompt congelado em 256² no modo `positive`
(τ = 0,5) e MarkerUNet com alpha = máscara do Cellpose. Só a perda muda entre os candidatos.

**Regras da varredura:**
1. **Os candidatos são definidos antes de rodar** (célula 2). Acrescentar candidatos depois de ver os resultados é sobreajuste à
   validação.
2. **O teste não é usado aqui.** A escolha é pela validação cruzada; o teste roda depois, uma vez, só para a configuração escolhida
   (com o notebook de experimento correspondente).
3. **Diferença pequena é empate:** se a diferença pareada para a referência tiver intervalo de confiança que cruza o zero, ou for
   menor que `EMPATE` em módulo, os candidatos são tratados como empatados, e vale o mais simples.

**Os resultados sobrevivem a uma desconexão:** cada candidato é gravado assim que termina (no Google Drive, no Colab) e, ao rodar de
novo, os candidatos já gravados são pulados. A tabela final lê tudo do disco.

**Saída:** uma pasta por candidato (configuração e procedência, métricas por imagem e por *fold*, curvas) em `RESULTS_DIR`, a tabela
`varredura_resumo.csv` e, no fim, uma cópia em `docs/estudo/resultados/varredura/` e um zip para baixar. Use um ambiente com **GPU**.

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração e onde gravar

In [ ]:
import csv
import gc
import json
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import load_preprocessed
from src.evaluation.report import save_run, summarize
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm, DMapTerm, RMSETerm, SizeTerm, TVTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.training.kfold import TrainConfig, run_kfold

RGBA_ALPHA = "mask"
SP_INPUT_SIZE = (256, 256)
TAU = 0.5
EMPATE = 0.01                # |diferença| abaixo disto conta como empate
REFERENCIA = "dice"          # candidato contra o qual os outros são comparados
MODELS_DIR = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))

# Mesma configuração dos notebooks de experimento: os resultados se comparam diretamente com eles.
CONFIG = TrainConfig(
    k=5, batch_size=4, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=20,
)

# No Colab, grava no Google Drive para sobreviver a uma desconexão do ambiente.
USAR_DRIVE = "google.colab" in sys.modules
if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/cell-fuzzy-seg/varredura"
else:
    RESULTS_DIR = os.path.join("docs", "estudo", "resultados", "varredura")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Device:", CONFIG.device, "| resultados em:", RESULTS_DIR)

## 2. Candidatos (definidos antes de rodar)

Um fator por vez em volta do Dice puro, mais o RMSE e a combinação da tese. Pesos escolhidos pela escala medida de cada termo
(`docs/estudo/09-correcoes-pontuais.md` §18.13): com peso 1, para um marcador do tamanho do núcleo, o Size vale ~1, o DMap ~0,6 e
o TV ~200; a perda de Dice convergida fica em ~0,2.

In [ ]:
CANDIDATOS = {
    "dice":       ("soft Dice",                          lambda: [DiceTerm()]),
    "tv_3e-4":    ("Dice + TV 1/3000 (proporção da tese)", lambda: [DiceTerm(), TVTerm(weight=1 / 3000)]),
    "tv_1e-3":    ("Dice + TV 0,001 (o do base_dice_tv)",  lambda: [DiceTerm(), TVTerm(weight=0.001)]),
    "size_0.03":  ("Dice + Size 0,03",                   lambda: [DiceTerm(), SizeTerm(weight=0.03)]),
    "size_0.1":   ("Dice + Size 0,1 (proporção da tese)", lambda: [DiceTerm(), SizeTerm(weight=0.1)]),
    "dmap_0.03":  ("Dice + DMap 0,03",                   lambda: [DiceTerm(), DMapTerm(weight=0.03)]),
    "dmap_0.1":   ("Dice + DMap 0,1",                    lambda: [DiceTerm(), DMapTerm(weight=0.1)]),
    "rmse":       ("RMSE no lugar do Dice",              lambda: [RMSETerm()]),
    "tese":       ("Dice + Size 0,1 + DMap 1/3000 + TV 1/3000 (pesos da tese / λ_seg)",
                   lambda: [DiceTerm(), SizeTerm(weight=0.1), DMapTerm(weight=1 / 3000), TVTerm(weight=1 / 3000)]),
}
assert REFERENCIA in CANDIDATOS
for nome, (descricao, termos) in CANDIDATOS.items():
    print(f"{nome:10s} {[(t.name, getattr(t._loss, 'weight', None)) for t in termos()]}")

## 3. Dados e rede final (compartilhada)

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA)
TRAIN_IDS = sorted(train_samples)
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_DIR, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)
print(f"Treino: {len(train_samples)} imagens")


def make_factory(termos):
    def factory():
        marker_net = MarkerUNet(config=UNET_CONFIG)
        pipeline = TrainingPipeline([
            MarkerStep(model=marker_net, differentiable=True, device=CONFIG.device),
            FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device),
        ])
        return marker_net, pipeline, LossComposer(termos())
    return factory

## 4. Varredura

Roda, em ordem, os candidatos que ainda não têm pasta em `RESULTS_DIR`, e grava cada um assim que termina. Para retomar depois de
uma desconexão, rode o notebook de novo: os candidatos prontos são pulados.

In [ ]:
for nome, (descricao, termos) in CANDIDATOS.items():
    if os.path.exists(os.path.join(RESULTS_DIR, nome, "resumo.json")):
        print(f"[{nome}] já gravado; pulando.")
        continue
    if os.path.exists(os.path.join(RESULTS_DIR, nome)):
        shutil.rmtree(os.path.join(RESULTS_DIR, nome))  # gravação incompleta de uma execução interrompida
    print(f"\n===== {nome}: {descricao} =====")
    folds = run_kfold(make_factory(termos), train_samples, TRAIN_IDS, CONFIG)
    save_run(nome, CONFIG, folds, description={
        "perdas": descricao, "varredura": True, "rgba_alpha": RGBA_ALPHA,
        "rede_final": f"ScribblePrompt-UNet v1 congelado, positive, tau={TAU}, input_size={SP_INPUT_SIZE}",
    }, output_dir=RESULTS_DIR)
    print(f"[{nome}] Dice de validação (média por imagem): {np.mean([r['dice'] for f in folds for r in f.rows]):.4f}")
    del folds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Tabela comparativa (lida do disco)

Para cada candidato gravado:
- **Dice de validação** (média por imagem das 37 predições fora do treino) e a comparação com o Cellpose;
- **diferença pareada por imagem contra a referência**, com IC 95% por *bootstrap* e Wilcoxon, e se é **empate**;
- **época pela curva média** (a forma usual de escolher um hiperparâmetro na validação cruzada): a época que maximiza o Dice médio
  entre os *folds*, procurada até a menor época em que algum *fold* parou, e o Dice médio nessa época. A diferença para o Dice
  acima mostra quanto a escolha da melhor época por *fold* deixa o número otimista.

In [ ]:
def ler_csv(caminho):
    with open(caminho, encoding="utf-8") as f:
        return list(csv.DictReader(f))


def curva_media(curvas):
    por_fold = {}
    for linha in curvas:
        if linha["val_dice"] != "":
            por_fold.setdefault(linha["fold"], {})[int(linha["epoch"])] = float(linha["val_dice"])
    ultima = min(max(c) for c in por_fold.values())
    medias = {e: np.mean([c[e] for c in por_fold.values()]) for e in range(1, ultima + 1)}
    melhor = max(medias, key=medias.get)
    return melhor, medias[melhor], ultima


prontos = [n for n in CANDIDATOS if os.path.exists(os.path.join(RESULTS_DIR, n, "resumo.json"))]
dice_por_imagem = {n: {r["id"]: float(r["dice"]) for r in ler_csv(os.path.join(RESULTS_DIR, n, "folds.csv"))} for n in prontos}
cellpose = {r["id"]: float(r["cellpose_dice"]) for r in ler_csv(os.path.join(RESULTS_DIR, prontos[0], "folds.csv"))}

tabela = []
for nome in prontos:
    ids = sorted(dice_por_imagem[nome])
    contra_cellpose = summarize([{"dice": dice_por_imagem[nome][i], "cellpose_dice": cellpose[i]} for i in ids], metrics=["dice"])["dice"]
    linha = {
        "candidato": nome,
        "perdas": CANDIDATOS[nome][0],
        "dice_val": contra_cellpose["mean"],
        "dice_val_std": contra_cellpose["std"],
        "dif_cellpose": contra_cellpose["diff_mean"],
    }
    if REFERENCIA in dice_por_imagem and nome != REFERENCIA:
        ref = dice_por_imagem[REFERENCIA]
        s = summarize([{"dice": dice_por_imagem[nome][i], "ref_dice": ref[i]} for i in ids], metrics=["dice"],
                      baseline_prefix="ref_")["dice"]
        linha.update({
            "dif_ref": s["diff_mean"], "dif_ref_ic95": f"{s['diff_ci95'][0]:+.4f} a {s['diff_ci95'][1]:+.4f}",
            "melhora_piora": f"{s['n_better']}/{s['n_worse']}", "wilcoxon_p": s["wilcoxon_p"],
            "empate": bool(s["diff_ci95"][0] <= 0 <= s["diff_ci95"][1] or abs(s["diff_mean"]) < EMPATE),
        })
    resumo_folds = ler_csv(os.path.join(RESULTS_DIR, nome, "folds_resumo.csv"))
    epoca, dice_curva, ultima = curva_media(ler_csv(os.path.join(RESULTS_DIR, nome, "curvas.csv")))
    linha.update({
        "mediana_melhor_epoca": float(np.median([int(r["best_epoch"]) for r in resumo_folds])),
        "epoca_curva_media": epoca, "dice_curva_media": dice_curva, "curva_ate": ultima,
    })
    tabela.append(linha)

tabela.sort(key=lambda r: -r["dice_val"])
caminho = os.path.join(RESULTS_DIR, "varredura_resumo.csv")
with open(caminho, "w", newline="", encoding="utf-8") as f:
    colunas = []
    for r in tabela:
        colunas += [c for c in r if c not in colunas]
    writer = csv.DictWriter(f, fieldnames=colunas)
    writer.writeheader()
    writer.writerows(tabela)

print(f"Referência: {REFERENCIA} | Cellpose (mesmas imagens): {np.mean(list(cellpose.values())):.4f}\n")
for r in tabela:
    ref = "" if "dif_ref" not in r else (f" | Δ ref {r['dif_ref']:+.4f} ({r['dif_ref_ic95']}) {r['melhora_piora']}"
                                          f"{' EMPATE' if r['empate'] else ''}")
    print(f"{r['candidato']:10s} Dice {r['dice_val']:.4f} ± {r['dice_val_std']:.4f} | Δ Cellpose {r['dif_cellpose']:+.4f}{ref}"
          f" | curva média: época {r['epoca_curva_media']} → {r['dice_curva_media']:.4f} (mediana {r['mediana_melhor_epoca']:.0f})")
print("\nTabela salva em:", caminho)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.5 * len(tabela) + 2))
nomes = [r["candidato"] for r in tabela][::-1]
medias = [r["dice_val"] for r in tabela][::-1]
erros = [r["dice_val_std"] / np.sqrt(len(cellpose)) for r in tabela][::-1]
ax.errorbar(medias, range(len(nomes)), xerr=erros, fmt="o")
ax.axvline(np.mean(list(cellpose.values())), color="k", ls="--", label="Cellpose")
ax.set_yticks(range(len(nomes)))
ax.set_yticklabels(nomes)
ax.set_xlabel("Dice de validação (média ± erro padrão sobre as imagens)")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "varredura.png"), dpi=80, bbox_inches="tight")
plt.show()

## 6. Copiar para o repositório e baixar (Colab)

In [ ]:
destino = os.path.join("docs", "estudo", "resultados", "varredura")
if os.path.abspath(RESULTS_DIR) != os.path.abspath(destino):
    shutil.copytree(RESULTS_DIR, destino, dirs_exist_ok=True)
    print("Copiado para", destino)
if "google.colab" in sys.modules:
    from google.colab import files

    archive = shutil.make_archive("/content/varredura", "zip", root_dir=os.path.dirname(os.path.abspath(destino)),
                                  base_dir="varredura")
    files.download(archive)  # descompactar em docs/estudo/resultados/